# Model Intercomparison Notebook

This notebook aggregates and visualizes evaluation artifacts from multiple model output directories.

Artifacts expected in each model folder (e.g. `output/verification_esfm/<model_name>`):

- `deterministic/metrics.csv`, `metrics_standardized.csv`
- `ets/ets_metrics.csv` or `ets_metrics_by_lead_wide.csv`
- `energy_spectra/*_energy.npz`
- `histograms/*_sfc_latbands_combined.npz`
- `wd_kde/*_sfc_latbands_kde_combined.npz`
- `maps/*.npz` (Ground truth + model arrays)
- `probabilistic/crps_summary.csv`, `*_pit_hist.npz`, `crps_map_*.npz`, optional WBX outputs

The helper functions in `swissclim_evaluations.intercompare` will combine common files across models and generate comparison plots & combined CSVs under an output root (default: `output/intercomparison`).

## 1. Configuration

Edit the lists below to point to the per-model evaluation output folders you want to intercompare. Optionally provide human-readable labels (must match length of `model_dirs`). If `labels` is left empty or length mismatch occurs, folder basenames are used.

In [ ]:
from pathlib import Path
from swissclim_evaluations import intercompare

# List paths to model evaluation output folders (each containing deterministic/, probabilistic/, etc.)
# Example: if you ran evaluations into output/verification_esfm/modelA, modelB, etc.:
# Always anchor relative paths to the project root (parent of notebooks) for consistency
project_root = Path.cwd().parent
model_dirs = [
    (project_root / "output/modelA").resolve(),
    (project_root / "output/modelB").resolve(),
    # Add more as needed
]

# Optional labels (same order as model_dirs) — leave empty to auto-derive from folder name
labels = [
    "Model A",
    "Model B",
]

# Output root for combined intercomparison artifacts (resolved relative to project root)
cfg_output = "output/notebook_intercomp"
out_root = (
    (project_root / cfg_output).resolve()
    if not cfg_output.startswith("/")
    else Path(cfg_output)
)
out_root.mkdir(parents=True, exist_ok=True)

# Sub-modules to run: choose any subset of: spectra, hist, kde, maps, metrics, prob
modules = ["spectra", "hist", "kde", "maps", "metrics", "prob"]

# Limit number of map panels to avoid huge output
max_map_panels = 4

model_dirs

## 2. Sanity Checks

We verify that each provided model directory exists and has at least one expected sub-folder. Warnings are printed for missing folders so you can correct paths before running the full intercomparison.

In [ ]:
for p in model_dirs:
    if not p.exists():
        print(f"WARNING: Missing model directory: {p}")
    else:
        subdirs = [d.name for d in p.iterdir() if d.is_dir()]
        print(f"Found {p} with subdirs: {subdirs}")

## 3. Run Intercomparison

This will:

- Generate comparison energy spectra plots (2D/3D) if common NPZ files found
- Overlay histogram distributions by latitude bands
- Overlay wind direction KDE curves by latitude bands
- Produce panel map comparisons (limit controlled by `max_map_panels`)
- Combine deterministic metrics (including standardized) and ETS metrics into CSVs & basic bar plots
- Combine probabilistic CRPS/PIT, CRPS maps, PIT hist overlaps, plus WBX temporal/spatial summaries if present

All combined artifacts are written under `out_root`. Existing files may be overwritten.

In [ ]:
# Derive labels if length mismatch
if not labels or len(labels) != len(model_dirs):
    labels = [p.name for p in model_dirs]
print("Using labels:", labels)

# Execute intercomparison modules
if "spectra" in modules:
    intercompare.intercompare_energy_spectra(model_dirs, labels, out_root)
if "hist" in modules:
    intercompare.intercompare_histograms(model_dirs, labels, out_root)
if "kde" in modules:
    intercompare.intercompare_wd_kde(model_dirs, labels, out_root)
if "maps" in modules:
    intercompare.intercompare_maps(
        model_dirs, labels, out_root, max_panels=max_map_panels
    )
if "metrics" in modules:
    intercompare.intercompare_metrics_csv(model_dirs, labels, out_root)
if "prob" in modules:
    intercompare.intercompare_probabilistic(model_dirs, labels, out_root)

print("Intercomparison complete. Artifacts written to", out_root.resolve())

## 4. Quick Look at Combined Metrics

If deterministic / probabilistic metrics were combined, load and display a few of them inline for inspection.

In [ ]:
import pandas as pd
from pathlib import Path

det_comb = out_root / "deterministic" / "metrics_combined.csv"
if det_comb.exists():
    df_det = pd.read_csv(det_comb)
    display(df_det.head())
else:
    print("Combined deterministic metrics CSV not found.")

prob_crps_comb = out_root / "probabilistic" / "crps_summary_combined.csv"
if prob_crps_comb.exists():
    df_crps = pd.read_csv(prob_crps_comb)
    display(df_crps.head())
else:
    print("Combined probabilistic CRPS summary not found.")

## 5. Exploring Combined Spatial / Temporal Probabilistic Metrics (WBX)

If WeatherBenchX WBX artifacts were produced, you can further slice / visualize them here.

In [ ]:
spatial_comb = out_root / "probabilistic" / "spatial_metrics_combined.csv"
temporal_comb = out_root / "probabilistic" / "temporal_metrics_combined.csv"
if spatial_comb.exists():
    df_spatial = pd.read_csv(spatial_comb)
    print("Spatial metrics columns:", df_spatial.columns.tolist()[:12], "...")
    display(df_spatial.head())
else:
    print("No combined spatial metrics CSV found.")
if temporal_comb.exists():
    df_temporal = pd.read_csv(temporal_comb)
    print("Temporal metrics columns:", df_temporal.columns.tolist()[:12], "...")
    display(df_temporal.head())
else:
    print("No combined temporal metrics CSV found.")